# CGLOPS Biophysical Products Explorer

This notebook demonstrates how to discover and visualise **CGLOPS biophysical products** served through the [Terrascope](https://terrascope.be/) STAC catalogue using the `rs_tools` package.

**V3 products:** NDVI, LAI, FAPAR  
**V2 products:** GPP, NPP, DMP, GDMP

We will:
1. List known products from the built-in catalog
2. Search the Terrascope archive for available data
3. Visualise seasonal "breathing pulse" animations
4. Compare complementary products with slider and RGB composites

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from rs_tools.config import SearchConfig, BoundingBox
from rs_tools.search import search_archive
from rs_tools.datasets.catalog import list_datasets, get
from rs_tools.visualization.timeseries import plot_timeseries_slider, plot_timeseries_line
from rs_tools.visualization.globe import add_globe_inset
from rs_tools.visualization.slider import slider_comparison
from rs_tools.visualization.rgb_composite import multi_temporal_rgb, plot_rgb

%matplotlib widget

## Known CGLOPS Products

List all CGLOPS products registered in the built-in catalog.

In [ ]:
cglops_datasets = list_datasets(tag="cglops")
for ds in cglops_datasets:
    print(f"{ds.short_name:25s} {ds.name}")

## Define Search Parameters

Set up a region of interest and temporal window for querying the archives.

In [ ]:
# Define a region of interest (example: Western Europe)
bbox = BoundingBox(west=-10, south=35, east=15, north=55)

config = SearchConfig(
    start_date="2020-01-01",
    end_date="2023-12-31",
    bbox=bbox,
    collections=[],  # will be populated per-product below
)

print(f"Search window: {config.date_range_str}")
print(f"Bounding box:  {bbox.as_tuple()}")

## Search Terrascope for V3 Products (NDVI, LAI, FAPAR)

Query the Terrascope STAC catalogue for each v3 biophysical product.

In [ ]:
# Search for each v3 product
v3_products = list_datasets(tag="v3")
for ds in v3_products:
    print(f"\n--- {ds.short_name} ---")
    collections = ds.archive_collections.get("terrascope", [])
    if collections:
        cfg = SearchConfig(
            start_date=config.start_date,
            end_date=config.end_date,
            bbox=config.bbox,
            collections=collections,
        )
        items = search_archive("terrascope", cfg)
        print(f"  Found {len(items)} items")
    else:
        print("  No Terrascope collection ID registered yet.")

## Search Terrascope for V2 Products (GPP, NPP, DMP, GDMP)

Query the Terrascope STAC catalogue for each v2 carbon/productivity product.

In [ ]:
# Search for each v2 product
v2_products = list_datasets(tag="v2")
for ds in v2_products:
    print(f"\n--- {ds.short_name} ---")
    collections = ds.archive_collections.get("terrascope", [])
    if collections:
        cfg = SearchConfig(
            start_date=config.start_date,
            end_date=config.end_date,
            bbox=config.bbox,
            collections=collections,
        )
        items = search_archive("terrascope", cfg)
        print(f"  Found {len(items)} items")
    else:
        print("  No Terrascope collection ID registered yet.")

## Visualization: Breathing Pulse Animation

Animate the seasonal "breathing pulse" of vegetation using NDVI time-series.
The slider lets you scrub through time steps interactively.

> **Note:** Synthetic placeholder data is used below. Replace with real Terrascope data once collection IDs are configured.

In [ ]:
# Synthetic NDVI-like data with seasonal cycle for demonstration
times = np.arange(
    np.datetime64("2020-01-01"),
    np.datetime64("2023-12-31"),
    np.timedelta64(10, "D"),
)
n_times = len(times)
y, x = np.meshgrid(np.linspace(0, 1, 100), np.linspace(0, 1, 100), indexing="ij")

# Simulate seasonal vegetation cycle
day_of_year = np.array([(t - np.datetime64(str(t)[:4])) / np.timedelta64(1, "D") for t in times])
seasonal = 0.3 + 0.4 * np.sin(2 * np.pi * day_of_year / 365 - np.pi / 2)
spatial_pattern = 0.5 + 0.3 * np.sin(2 * np.pi * y) * np.cos(2 * np.pi * x)

data_3d = seasonal[:, None, None] * spatial_pattern[None, :, :]
dummy = xr.DataArray(
    data_3d.astype(np.float32),
    dims=["time", "y", "x"],
    coords={"time": times},
)

fig = plot_timeseries_slider(dummy, title="NDVI Breathing Pulse (placeholder)", cmap="YlGn")
plt.show()

## Globe Inset

Add a 3-D orthographic globe showing the region of interest as context.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(dummy.isel(time=18).values, cmap="YlGn", origin="upper")
ax.set_title("NDVI — Western Europe (placeholder)")
add_globe_inset(fig, bbox)
plt.show()

## Product Comparison Slider

Compare two products side-by-side with a draggable divider.

In [ ]:
# Placeholder: NDVI vs LAI comparison
left = dummy.isel(time=18).values   # simulated "NDVI"
right = dummy.isel(time=36).values  # simulated "LAI"

fig = slider_comparison(
    left, right,
    left_label="NDVI",
    right_label="LAI",
    cmap="YlGn",
    title="NDVI vs LAI Comparison (placeholder)",
)
plt.show()

## Multi-Temporal RGB Composite

Assign three time steps to R, G, B channels to visualise seasonal change.
- **Red** = winter  
- **Green** = spring  
- **Blue** = summer

In [ ]:
# Multi-temporal composite: winter / spring / summer
# Indices: ~Jan (0), ~Apr (9), ~Jul (18) of first year
rgb = multi_temporal_rgb(dummy, time_indices=(0, 9, 18))
fig = plot_rgb(rgb, title="Multi-Temporal NDVI Composite (placeholder)")
plt.show()

## Next Steps

- Populate actual Terrascope collection IDs in the dataset catalog
- Download and cache actual CGLOPS raster data
- Build full breathing-pulse animations with real multi-year time-series
- Add location-specific slider comparisons for interesting regions (Amazon, Sahel, Europe)
- Export animations as GIF/MP4 for sharing